In [6]:
using Random
using LinearAlgebra
using Statistics
using Distributions 
using Optim 
using Combinatorics
using MLDataUtils
using ForwardDiff

In [32]:
# Esse é o caso Homoskedastico - ou seja, equivalente ao LIML

function cue_objective(beta::Vector, y::Matrix, X::Matrix, Z::Matrix)     # Q(b) = LIML(b)
    
    u = y - X * beta 
    u_hat = Z * (Z \ u) 

    numerator_matrix = u_hat' * u_hat
    denominator_matrix = u' * u                  
    
    return (numerator_matrix[1] / denominator_matrix[1])
end


# LIML quadratificada

function cue_objective2(beta::Vector, y::Matrix, X::Matrix, Z::Matrix, beta_liml::Vector)
    cue_to_h = b -> cue_objective(b, y, X, Z)
    D2Q = ForwardDiff.hessian(cue_to_h, beta_liml)
    obj = cue_to_h(beta_liml) +  (1/2)*(beta - beta_liml)'*D2Q*(beta - beta_liml)
    return obj
end


cue_objective2 (generic function with 1 method)

In [72]:
# Função objetivo com várias penalidades diferentes

function objective_penalty(beta::Vector, y::Matrix,
                           X::Matrix, Z::Matrix, 
                           lambda::Real,penalty_type::Integer,
                            beta_liml::Vector)
    
    obj = cue_objective(beta, y, X, Z) # Assume que cue_objective existe
    
    local penalty_term::Real

    # Pré-calcula as normas mais comuns para eficiência
    norm_b1 = sum(abs.(beta))     # Norma L1: ||β||₁
    norm_b2_sq = sum(beta.^2)   # Norma L2 ao quadrado: ||β||₂²

    h = norm(beta_liml,2)^2

    if penalty_type == 1 
        # L2 (Ridge)

        penalty_term = lambda * norm_b2_sq/h

    elseif penalty_type == 2
        
        penalty_term = lambda * 2 * (norm_b2_sq/h) / (1 + norm_b2_sq/h)  

    elseif penalty_type == 3
        
        penalty_term = lambda * tanh(norm_b2_sq/h)/tanh(1) 

    elseif penalty_type == 4
         
        penalty_term = lambda * (1 - exp(-norm_b2_sq/h)) / (1 - exp(-1)) 

    elseif penalty_type == 5
        
        let log_norm = log(1 + norm_b2_sq/h) # 'let' cria uma variável local
            penalty_term = lambda * (log_norm / (1 + log_norm)) / ( log(2)/(1 + log(2))   )
        end

    elseif penalty_type == 6
        obj = cue_objective2(beta, y, X, Z, beta_liml)           # forma quadratificada
        
        penalty_term = lambda * norm_b2_sq/h            

    elseif penalty_type == 7

        w = [ 1/abs(beta_liml[j]) for j in 1:length(beta_liml)]
        penalty_term = lambda * sum(  w  ) * sum( w .* (beta.^2) ./ (1 .+ beta.^2)   )    

    elseif penalty_type == 8
        h = norm(beta_liml,2)^2
        penalty_term = lambda * (4/pi * atan(norm_b2_sq/h))    
        
    elseif penalty_type == 9
        penalty_term = 0
        
    else 
        # Mensagem de erro atualizada
        error("Tipo de penalidade (penalty_type) inválido. Use um inteiro de 1 a 9.")
    end

    return obj + penalty_term
end

objective_penalty (generic function with 1 method)

In [73]:
function generate_beta_combinations(beta::AbstractVector{T}) where {T<:Real}     # gera combinações de chutes iniciais. 
    P = length(beta)
    n = 1 << P
    combos = Vector{Vector{T}}(undef, n)
    for s in 0:(n-1)
        v = zeros(T, P)
        @inbounds for j in 1:P
            if ((s >>> (j-1)) & 0x1) == 1
                v[j] = beta[j]
            end
        end
        combos[s+1] = v
    end
    return combos
end


generate_beta_combinations (generic function with 1 method)

In [74]:
function loss_mse(beta::Vector, y::Matrix, X::Matrix)    # estima o MSE 
    return sum((y - X*beta).^2)
end

loss_mse (generic function with 1 method)

In [75]:
function LIML(y::AbstractMatrix, X::AbstractMatrix, Z::AbstractMatrix)    # encontra o estimador de LIML
    Y = hcat(y, X)
    YtZ = Y' * Z
    temp = Z\Y
    A = YtZ * temp
    B = Y' * Y
    vals, vecs = eigen(A, B)
    min_idx = argmin(vals)
    v = vecs[:, min_idx]
    return -v[2:end] ./ v[1]
end

function tsls(y::AbstractMatrix, X::AbstractMatrix, Z::AbstractMatrix)    # encontra o estimador de TSLS
    X_hat = Z * (Z \ X)
    beta_2sls = X_hat \ y
    return beta_2sls
end

function Q_ext(y::AbstractMatrix, X::AbstractMatrix, Z::AbstractMatrix)
    Y = hcat(y, X)
    YtZ = Y' * Z
    temp = Z\Y
    A = YtZ * temp
    B = Y' * Y
    vals, vecs = eigen(A, B)
    return minimum(vals), maximum(vals)           # dá a imagem da função objetivo
end



function optimizador(y::Matrix, X::Matrix, Z::Matrix, lambda::Real, pen_type::Integer, combinacoes,     # otimiza a função objetivo penalizada
                     beta_liml::Vector)
                    #beta_liml::Vector, beta_ols::Matrix)
    
    func_to_minimize = b -> objective_penalty(b, y, X, Z, 
                                              lambda, pen_type,beta_liml)  #, beta_liml, beta_ols)

    best_solution = nothing
    min_objective_value = Inf

    for (i, start_point) in enumerate(combinacoes)
        results = optimize(func_to_minimize, start_point, LBFGS())
        if Optim.converged(results)
            current_min = Optim.minimum(results)
                if current_min < min_objective_value
                    # Atualiza a melhor solução
                    min_objective_value = current_min
                    best_solution = Optim.minimizer(results)
                end
        end
    end
    return best_solution

end


optimizador (generic function with 1 method)

In [76]:
function round_first_digit(x::Real)        # arredondamento 
    x == 0 && return 0.0
    exp10 = floor(log10(abs(x)))         # expoente do primeiro dígito
    scale = 10.0^exp10
    return round(x / scale; digits=1) * scale
end

round_first_digit (generic function with 1 method)

In [77]:
function find_best_lambda_and_refit(y::Matrix, X::Matrix, Z::Matrix,
                                    fold_iter, lambdas_to_test::AbstractVector,                           # encontra o lambda ótimo por CV e o estimador associado
                                    pen_type::Integer, combinacoes, beta_liml::Vector)

    errors_by_lambda = Dict{Float64, Vector{Float64}}()
    for lambda in lambdas_to_test
        errors_by_lambda[lambda] = Float64[]
    end

    # 1. Loop de Cross-Validation (K-Folds)
    for (train_indices, test_indices) in fold_iter
        
        # 1a. Separar dados do fold
        y_train, y_test = y[train_indices, :], y[test_indices, :]
        X_train, X_test = X[train_indices, :], X[test_indices, :]
        Z_train = Z[train_indices, :]


        # 1c. Loop de Grid Search (Lambdas)
        for lambda in lambdas_to_test
            
            # Otimizar com dados de TREINO
            best_solution = optimizador(y_train, X_train, Z_train, lambda, pen_type,
                                        combinacoes, beta_liml)#, beta_liml, beta_ols) 
            
            # Avaliar com dados de TESTE
            if best_solution == nothing
                loss = Inf
                push!(errors_by_lambda[lambda], loss)
            else    
                loss = loss_mse(best_solution, y_test, X_test)
                # Armazenar o erro
                push!(errors_by_lambda[lambda], loss)
            end
                
        end
    end

    # 2. Encontrar o Melhor Lambda
    # Calcular o erro médio de CV para cada lambda
    avg_errors = Dict{Float64, Float64}()
    for (lambda, errors) in errors_by_lambda
        avg_errors[lambda] = mean(errors)
    end

    # Encontrar o lambda que minimizou o erro médio
    best_pair = findmin(avg_errors) # Retorna (min_erro, lambda_com_min_erro)
    best_lambda = best_pair[2]

    best_solution_final = optimizador(y, X, Z, best_lambda, pen_type, combinacoes,
                                      beta_true)

    # Retorna o lambda escolhido e a solução final
    return best_lambda, best_solution_final
end

find_best_lambda_and_refit (generic function with 1 method)

# Simulações

In [134]:
function simulate_iv(T; β0, ρ, φ, π, m)
    z1 = randn(T)
    v  = randn(T)
    θ2 = randn(T)
    θ1 = abs.(z1) .* randn(T)       # N(0, z1_t^2)

    ε = ρ .* v .+ sqrt(1 - ρ^2) .* (φ .* θ1 .+ sqrt(1 - φ^2) .* θ2)

    X = π .* z1 .+ v
    y = β0 .* X .+ ε

    # Z = [1, z1, z1^2, z1^3, z1^4, z1*D1, ...]
    Z = ones(T, m)
    npowers = 4
    for j in 0:npowers
        Z[:, j+1] = z1 .^ j
    end
    for k in 6:m
        Dk = rand(T) .< 0.5                      # Bool ~ Bernoulli(0.5)
        Z[:, k] = z1 .* Float64.(Dk)
    end

    return y, X, Z
end

# exemplo de uso (sem seed):
N = 400

y, X, Z = simulate_iv(N; β0= 0.0, ρ=0.3, φ=0.5, π=sqrt(8), m=30)

([-0.16311927541360888, -0.5884187584451741, 1.306329681776904, 0.33673066502215554, -0.31018208353216065, -1.8659615377827574, -0.028659842131689595, -0.4710723744759718, 1.3764205281973059, -0.010890649839186195  …  -0.5791689256276951, -0.14392955409951697, -0.635318093579955, 0.42433773280024534, 1.4392643528512008, 0.28554388879081505, 0.4358086703794479, -0.09441393433133502, 0.38177195239768513, -0.2561202938774616], [-1.348825148233436, 3.4194886729047003, -4.124350184260215, -0.6098718097478193, 2.4535047029090693, -8.843786120349273, -0.3975438212113319, -1.8978517127118166, 2.7888674340600663, 1.788155481520393  …  -2.450789917107123, -4.412528228716053, -1.3307994687753089, 3.254318318973546, 2.291943358635329, -2.533965751685801, -3.7410109400699514, -1.7241460535980564, 2.230414000389846, -1.8708897193449043], [1.0 -0.5134411035957492 … -0.0 -0.5134411035957492; 1.0 1.141798879786276 … 1.141798879786276 0.0; … ; 1.0 0.8410790547696622 … 0.0 0.0; 1.0 -0.8569540606061715 … 

In [137]:
## Initial settings
Random.seed!(3131)

N = 400  # Número de observações
P = 1   # Número de endogenas
L = 30    # Número de instrumentos, 5, 10, 30, 50
                     
beta_true = [10]
Pi = sqrt(8)

R = 3   # Número de simulações 
K = 2      # Número de folds

# divisors = [1]

lambdas_0 = [  ]    # obrigo a escolher Qmax - Qmin / N     (acho que isso é o lambda*k do Miguel)

fold_iter = kfolds(1:N, k=K)
num_penalties = 9


9

In [138]:
results = zeros(R, P, num_penalties+2)
best_lambdas_storage = zeros(R, num_penalties)

for r in 1:R
    rng = Random.Xoshiro(10+r+101)

    y, X, Z = simulate_iv(N; β0 = beta_true[1], ρ=0.3, φ= 0.5, π = Pi, m = L)

    y, X = reshape(y, :, 1) , reshape(X, :, 1)
    
    
    # Build a lambda that makes sense with the problem

    lambdas = lambdas_0
    
    Qmin, Qmax = Q_ext(y,X,Z)
    diff = (Qmax - Qmin) # /N

    # lambdas = [diff/d for d in divisors]             # o tabak usava um grid diferente para cada
    # push!(lambdas, 1.0)
    # push!(lambdas, 0.0) 
    

    push!(lambdas, diff)
         
    
    
    beta_liml = LIML(y, X, Z)
    beta_2sls = tsls(y,X,Z)[:,1]
    combinacoes = generate_beta_combinations(beta_2sls)    

    
    for pen_type in 1:num_penalties
        (lambda_escolhido, solucao_final) = find_best_lambda_and_refit(
            y, X, Z, 
            fold_iter, 
            lambdas, 
            pen_type, combinacoes, beta_2sls
        )
        
        results[r, :, pen_type] = solucao_final
        
        best_lambdas_storage[r, pen_type] = lambda_escolhido
    end
    
    results[r, :, num_penalties+1] = beta_liml
    results[r, :, num_penalties+2] = beta_2sls
    
    print("\r... completou a replicação de Monte Carlo $r/$R")
    flush(stdout)


end

... completou a replicação de Monte Carlo 3/3

In [139]:

# --- 1. Preparar 'beta_true' ---
beta_true_row = beta_true' 

# --- 2. Storage para todas as métricas ---
all_mean_biases = zeros(num_penalties, P)
all_median_biases = zeros(num_penalties, P)
all_mse_per_param = zeros(num_penalties, P) 
all_percentile_01 = zeros(num_penalties, P)
all_percentile_99 = zeros(num_penalties, P)

# Métricas escalares resumidas (Vetores [pen_type])
all_mean_abs_biases = zeros(num_penalties)
all_mean_abs_median_biases = zeros(num_penalties)
all_mean_mse = zeros(num_penalties)

# --- NOVAS MÉTRICAS ESCALARES ---
# Média do Range Inter-Percentil (P99-P01)
all_mean_ipr = zeros(num_penalties)
# "Média das Médias" (Média dos vieses médios)
all_mean_of_means = zeros(num_penalties)
# --- FIM NOVAS MÉTRICAS ---


# --- 3. Loop por cada penalidade para calcular as métricas ---
for pen_type in 1:num_penalties
    
    all_betas_p = results[:, :, pen_type]
    biases_matrix = all_betas_p .- beta_true_row
    
    # --- Viés Médio (Mean Bias), MAB e "Média das Médias" ---
    mean_bias_vector = mean(biases_matrix, dims=1)
    
    all_mean_biases[pen_type, :] = mean_bias_vector # Armazena o vetor (1, P)
    all_mean_abs_biases[pen_type] = mean(abs.(mean_bias_vector)) # MAB
    
    # ADICIONADO: "Média das Médias"
    all_mean_of_means[pen_type] = mean(mean_bias_vector) 
    
    # --- Mediana do Viés & Média da Mediana Absoluta ---
    median_bias_vector = median(biases_matrix, dims=1)
    all_median_biases[pen_type, :] = median_bias_vector # Armazena o vetor (1, P)
    all_mean_abs_median_biases[pen_type] = mean(abs.(median_bias_vector))

    # --- Média das Médias Quadráticas (Mean MSE) ---
    squared_biases_matrix = biases_matrix.^2
    mse_vector = mean(squared_biases_matrix, dims=1)
    all_mse_per_param[pen_type, :] = mse_vector # Armazena o vetor (1, P)
    all_mean_mse[pen_type] = mean(mse_vector) 

    # --- Quartis 1 e 99 (Percentis) ---
    p01_vector = mapslices(v -> quantile(v, 0.01), biases_matrix, dims=1)
    p99_vector = mapslices(v -> quantile(v, 0.99), biases_matrix, dims=1)
    
    all_percentile_01[pen_type, :] = p01_vector
    all_percentile_99[pen_type, :] = p99_vector
    
    # ADICIONADO: Resumo escalar dos percentis (Média do Range P99-P01)
    # ipr_vector (Inter-Percentile Range) é um vetor (1, P)
    ipr_vector = p99_vector .- p01_vector
    all_mean_ipr[pen_type] = mean(ipr_vector)
    
end

# --- 4. Exibir os resultados ---
println("--- Análise Detalhada de Métricas por Penalidade ---")

for pen_type in 1:num_penalties
    println("\n" * "="^40)
    println("Resultados para Penalidade $pen_type")
    println("="^40)
    
    println("\n--- Métricas Escalares de Resumo ---")
    println("Média Viés Absoluto (MAB):     ", all_mean_abs_biases[pen_type])
    println("Média Mediana Absoluta (MAMB): ", all_mean_abs_median_biases[pen_type])
    println("Média do MSE:                  ", all_mean_mse[pen_type])
    println("Média do Range (P99-P01):      ", all_mean_ipr[pen_type])
    println("Média dos Vieses (M-de-M):     ", all_mean_of_means[pen_type])
end


--- Análise Detalhada de Métricas por Penalidade ---

Resultados para Penalidade 1

--- Métricas Escalares de Resumo ---
Média Viés Absoluto (MAB):     9.959859499741906
Média Mediana Absoluta (MAMB): 9.958609699840904
Média do MSE:                  99.1988340788353
Média do Range (P99-P01):      0.013588448951898258
Média dos Vieses (M-de-M):     -9.959859499741906

Resultados para Penalidade 2

--- Métricas Escalares de Resumo ---
Média Viés Absoluto (MAB):     9.980022457114494
Média Mediana Absoluta (MAMB): 9.979402847486412
Média do MSE:                  99.60085631045644
Média do Range (P99-P01):      0.006735953748284729
Média dos Vieses (M-de-M):     -9.980022457114494

Resultados para Penalidade 3

--- Métricas Escalares de Resumo ---
Média Viés Absoluto (MAB):     9.969496823229017
Média Mediana Absoluta (MAMB): 9.968548998618619
Média do MSE:                  99.3908857888061
Média do Range (P99-P01):      0.010305737094682854
Média dos Vieses (M-de-M):     -9.96949682322901

In [149]:
z2 = vec( results[:,:,2] )
z6 = vec( results[:,:,6] )  

println(z2)
println(z6)

[0.01623102697540135, 0.02059715251358835, 0.023104449167528174]
[9.989318258630496, 9.986478705094871, 9.98924812066401]


# Simulação única

In [112]:

y, X, Z = simulate_iv(N; β0 = beta_true[1], ρ=0.3, φ= 0.5, π = Pi, m = L)

y, X = reshape(y, :, 1) , reshape(X, :, 1)


# Build a lambda that makes sense with the problem

lambdas = [ ] 

Qmin, Qmax = Q_ext(y,X,Z)
diff = (Qmax - Qmin) /N

push!(lambdas, diff)

beta_liml = LIML(y, X, Z)
beta_2sls = tsls(y,X,Z)[:,1]
combinacoes = generate_beta_combinations(beta_2sls)    


results = Vector{Any}(undef, num_penalties)


for pen_type in 1:num_penalties
    (lambda_escolhido, solucao_final) = find_best_lambda_and_refit(
        y, X, Z, 
        fold_iter, 
        lambdas, 
        pen_type, combinacoes, beta_2sls
    )
    
    results[pen_type] = solucao_final
    
    best_lambdas_storage[pen_type] = lambda_escolhido
end


results


9-element Vector{Any}:
 [9.983734210575138]
 [9.983746328248428]
 [9.983745050637589]
 [9.983744328590342]
 [9.98374812932117]
 [9.999976256757394]
 [9.983758522340128]
 [9.983742995654286]
 [9.983758524739098]